# Chapter 17 — PDF and Word Documents

Source: [Automate the Boring Stuff with Python, 3rd Ed. — Chapter 17](https://automatetheboringstuff.com/3e/chapter17.html)

**Setup note:** Needs `pip install pypdf python-docx`.

This notebook favors *reading* over *writing*: each section has a short explanation followed by real, runnable code. Read it, run it, tweak a value and re-run it — that's the fastest way to build reading fluency.

---


## 17.1 Extracting text from a PDF

`pypdf.PdfReader(path)` opens a PDF; `.pages[i].extract_text()` pulls the text from a given page. Extraction quality varies a lot depending on how the PDF was generated (scanned images need OCR instead — see the OCR notebook).


In [1]:
# from pypdf import PdfReader
# reader = PdfReader('report.pdf')
# print(len(reader.pages))
# print(reader.pages[0].extract_text())
print('PdfReader(path).pages[i].extract_text() is the core pattern to remember.')


PdfReader(path).pages[i].extract_text() is the core pattern to remember.


## 17.2 Merging and splitting PDFs

`PdfWriter` builds a new PDF by adding pages from one or more source PDFs — merge by adding every page from several files in order, or split by writing selected pages into separate output files.


In [2]:
# from pypdf import PdfReader, PdfWriter
# writer = PdfWriter()
# for path in ['part1.pdf', 'part2.pdf']:
#     reader = PdfReader(path)
#     for page in reader.pages:
#         writer.add_page(page)
# with open('merged.pdf', 'wb') as f:
#     writer.write(f)
print('Merging = collect pages from multiple readers into one PdfWriter, then write().')


Merging = collect pages from multiple readers into one PdfWriter, then write().


## 17.3 Reading and writing Word documents

`python-docx`'s `Document` object models a `.docx` file as a list of paragraphs (and tables). Reading loops over `doc.paragraphs` and reads `.text`; writing calls `doc.add_paragraph()` / `doc.add_heading()` and finishes with `doc.save()`.


In [3]:
from docx import Document

doc = Document()
doc.add_heading('Weekly Report', level=1)
doc.add_paragraph('Everything is on track.')
doc.save('demo_report.docx')

doc2 = Document('demo_report.docx')
for p in doc2.paragraphs:
    print(repr(p.text))

import os
os.remove('demo_report.docx')


'Weekly Report'
'Everything is on track.'


### 17.1b An alternative extractor: PdfMiner

`PyPDF`'s `.extract_text()` (used above) is fast and good enough for most PDFs, but its text extraction can occasionally scramble the reading order on complex layouts (multi-column pages, tables). The third-party `PdfMiner` (specifically its `pdfminer.six` package, imported as `pdfminer`) does deeper layout analysis and often extracts more faithfully-ordered text from tricky PDFs, at the cost of being slower. The book presents it as PyPDF's companion tool: try PyPDF first since it's faster, and reach for PdfMiner's `extract_text()` when PyPDF's output looks jumbled.

In [4]:
from reportlab.pdfgen import canvas
from pdfminer.high_level import extract_text

# Build a tiny real PDF to extract from
c = canvas.Canvas('pdfminer_demo.pdf')
c.drawString(100, 700, 'PdfMiner extracted this line.')
c.save()

text = extract_text('pdfminer_demo.pdf')
print(repr(text.strip()))

import os
os.remove('pdfminer_demo.pdf')


'PdfMiner extracted this line.'


## 17.4 Encrypting and decrypting PDFs

`pypdf`'s `PdfWriter` can password-protect a PDF: `writer.encrypt(password)` before saving requires that password to reopen the file in any PDF reader, not just in Python. To read an encrypted PDF back programmatically, check `reader.is_encrypted`, then call `reader.decrypt(password)` before extracting text or pages — attempting to read an encrypted PDF's content without decrypting first raises an error.

In [5]:
from reportlab.pdfgen import canvas
from pypdf import PdfReader, PdfWriter

c = canvas.Canvas('secret_demo.pdf')
c.drawString(100, 700, 'Confidential content.')
c.save()

reader = PdfReader('secret_demo.pdf')
writer = PdfWriter()
writer.append(reader)
writer.encrypt('sw0rdfish')
with open('secret_demo_encrypted.pdf', 'wb') as f:
    writer.write(f)

reader2 = PdfReader('secret_demo_encrypted.pdf')
print('is_encrypted:', reader2.is_encrypted)

reader2.decrypt('sw0rdfish')
print('after decrypt():', reader2.pages[0].extract_text().strip())

import os
os.remove('secret_demo.pdf')
os.remove('secret_demo_encrypted.pdf')


is_encrypted: True
after decrypt(): Confidential content.


## 17.5 Extracting images from a PDF

Beyond text, `pypdf` can pull out the embedded images themselves: each `page.images` is a list of image objects, and each one's `.data` holds the raw image bytes (which you can write straight to a file) while `.name` gives its original internal filename inside the PDF.

In [6]:
from reportlab.pdfgen import canvas
from PIL import Image
from pypdf import PdfReader

# Build a tiny real PDF that embeds an actual image
Image.new('RGB', (40, 40), color='blue').save('tiny_square.png')
c = canvas.Canvas('image_demo.pdf')
c.drawImage('tiny_square.png', 100, 600, width=40, height=40)
c.save()

reader = PdfReader('image_demo.pdf')
page = reader.pages[0]
print('images found on page:', len(page.images))

for img in page.images:
    print('extracted image name:', img.name, '-- bytes:', len(img.data))
    with open('extracted_' + img.name, 'wb') as f:
        f.write(img.data)

import os
os.remove('tiny_square.png')
os.remove('image_demo.pdf')
for img in reader.pages[0].images:
    if os.path.exists('extracted_' + img.name):
        os.remove('extracted_' + img.name)


images found on page: 1
extracted image name: FormXob.79d14b2b5fe58f7b5b23d1e6cdd963a0.png -- bytes: 103
